In [7]:
import pandas as pd
import numpy as np
import altair as alt
from IPython.display import display

# Load the data
df = pd.read_csv('finchat-dashboard.csv')

# Replace '2330' with 'TSMC' in the 'Ticker' column
df['Ticker'] = df['Ticker'].replace('2330', 'TSMC')

# ===================================================
# 1. ROBUST DATA CLEANING FUNCTIONS
# ===================================================

def clean_percentage(series):
    """Removes '%' and converts to float."""
    return (series.astype(str)
                  .str.replace('%', '', regex=False)
                  .str.strip()
                  .replace(r'—', np.nan, regex=True)
                  .astype(float, errors='ignore'))

def clean_currency(series):
    """Removes currency symbols and commas (US/UK style)."""
    return (series.astype(str)
                  .str.replace('€', '', regex=False)
                  .str.replace('$', '', regex=False)
                  .str.replace('NT$', '', regex=False)
                  .str.replace(',', '', regex=False)
                  .str.strip()
                  .replace(r'—', np.nan, regex=True)
                  .astype(float, errors='ignore'))

def clean_market_cap(series):
    """Handles magnitude suffixes (M, B, T) and returns formatted strings."""
    def convert_cap(value):
        # Ensure value is a string before calling replace
        value = str(value).strip().upper().replace('€', '').replace('$', '').replace('NT$', '')
        if not value or value == '—':
            return np.nan

        # Identify suffix
        suffix = ''
        if value.endswith('T'):
            suffix = 'T'
            num_part = value[:-1]
        elif value.endswith('B'):
            suffix = 'B'
            num_part = value[:-1]
        elif value.endswith('M'):
            suffix = 'M'
            num_part = value[:-1]
        else:
            num_part = value # No suffix

        # Clean number part: remove thousands separators and ensure decimal is a period
        # Handle cases like "1,234.56" (US/UK) and "1.234,56" (European)
        if '.' in num_part and ',' in num_part:
            # If both are present, assume period is thousands separator and comma is decimal separator (European)
            num_part = num_part.replace('.', '').replace(',', '.')
        else:
            # Otherwise, assume comma is thousands separator and period is decimal separator (US/UK)
            num_part = num_part.replace(',', '')


        try:
            num = float(num_part)
        except ValueError:
            return np.nan

        # Apply magnitude and format as string based on original suffix or determined magnitude
        if suffix == 'T' or num >= 1e12:
            return f'{num/1e12:.2f}T'
        elif suffix == 'B' or num >= 1e9:
            return f'{num/1e9:.2f}B'
        elif suffix == 'M' or num >= 1e6:
            return f'{num/1e6:.2f}M'
        else:
            if num >= 1e12:
                return f'{num/1e12:.2f}T'
            elif num >= 1e9:
                return f'{num/1e9:.2f}B'
            elif num >= 1e6:
                return f'{num/1e6:.2f}M'
            else:
                return f'{num:.2f}'

    return series.apply(convert_cap)

# Apply Cleaning
df['Ownership Performance Clean'] = clean_percentage(df['Ownership Performance'])
df['Portfolio Percentage Clean'] = clean_percentage(df['Portfolio Percentage'])
df['P/E Clean'] = df['P/E'].replace('—', np.nan).astype(float, errors='ignore')
df['Market Cap Clean Formatted'] = clean_market_cap(df['Market Cap']) # Create a new column for formatted strings
df['Market Value Clean'] = clean_currency(df['Market Value']) # Used for filtering

# Filter the data for each chart type
df_valued = df.dropna(subset=['Portfolio Percentage Clean', 'Ownership Performance Clean']).copy()
# Use the new formatted column for the scatter plot
df_scatter = df.dropna(subset=['Ticker', 'Ownership Performance Clean', 'P/E Clean']).copy() # Removed Market Cap from filter


# ===================================================
# 2. CHART 1: PORTFOLIO ALLOCATION (PIE CHART)
# ===================================================

base_pie = alt.Chart(df_valued).encode(
    theta=alt.Theta("Portfolio Percentage Clean", stack=True)
).properties(
    title='Portfolio Allocation by Ticker'
)

pie = base_pie.mark_arc(outerRadius=120).encode(
    color=alt.Color("Ticker"),
    order=alt.Order("Portfolio Percentage Clean", sort="descending"),
    tooltip=["Ticker", alt.Tooltip("Portfolio Percentage Clean", title="Portfolio %", format=".1f")]
)

text = base_pie.mark_text(radius=140).encode(
    text=alt.Text("Portfolio Percentage Clean", format=".1f"),
    order=alt.Order("Portfolio Percentage Clean", sort="descending"),
    color=alt.value("black")
)

chart_allocation_pie = (pie + text).interactive()
print("--- 1. Portfolio Allocation Pie Chart ---")
display(chart_allocation_pie)


# ===================================================
# 3. CHART 2: TOTAL RETURNS (BAR CHART)
# ===================================================

chart_total_returns = alt.Chart(df_valued).mark_bar().encode(
    x=alt.X('Ticker', sort=alt.EncodingSortField(field='Ownership Performance Clean', op='sum', order='descending'), title='Ticker'),
    y=alt.Y('Ownership Performance Clean', title='Total Return (Ownership Performance %)'),
    color=alt.condition(
        alt.datum['Ownership Performance Clean'] < 0,
        alt.value('red'),
        alt.value('green')
    ),
    tooltip=[
        'Ticker',
        alt.Tooltip('Ownership Performance Clean', title='Total Return %', format='.1f')
    ]
).properties(
    title='Total Returns (Ownership Performance) by Ticker'
).interactive()

print("\n--- 2. Total Returns Bar Chart ---")
display(chart_total_returns)


# ===================================================
# 4. PERFORMANCE vs. VALUATION (SCATTER PLOT)
# ===================================================

chart_performance_valuation_scatter = alt.Chart(df_scatter).mark_circle(size=100).encode( # Increased size of the markers
    x=alt.X('P/E Clean', title='Valuation (P/E Ratio)'),
    y=alt.Y('Ownership Performance Clean', title='Total Return (Ownership Performance %)'),
    color=alt.Color('Ticker', legend=alt.Legend(title="Ticker")),
    tooltip=[
        'Ticker',
        alt.Tooltip('P/E Clean', title='P/E Ratio', format='.1f'),
        alt.Tooltip('Ownership Performance Clean', title='Total Return %', format='.1f'),
    ]
).properties(
    title='Total Return vs. Valuation (P/E Ratio) by Ticker'
)

# Add Ticker labels to the scatter plot
chart_performance_valuation_scatter = chart_performance_valuation_scatter + chart_performance_valuation_scatter.mark_text(
    align='left',
    baseline='middle',
    dx=10,
    text='Ticker'
).encode(
    size=alt.value(0),
    color=alt.value('black')
).interactive()

print("\n--- 3. Performance vs. Valuation Scatter Plot ---")
display(chart_performance_valuation_scatter)

# ===================================================
# 5. CHART 4: ALLOCATION IMPACT VS. RETURNS (SCATTER PLOT)
# ===================================================
chart_allocation_impact = alt.Chart(df_valued).mark_circle().encode( # Use df_valued as it contains Portfolio Percentage Clean
    # X-axis: Position Size
    x=alt.X('Portfolio Percentage Clean', title='Position Size (Portfolio Allocation %)'),
    # Y-axis: Total Return
    y=alt.Y('Ownership Performance Clean', title='Total Return (Ownership Performance %)'),

    # Dot size based on Market Value to show absolute dollar impact
    size=alt.Size('Market Value Clean',
                  title='Market Value (€)',
                  scale=alt.Scale(range=[50, 800])),

    color=alt.Color('Ticker', legend=alt.Legend(title="Ticker")),

    # Tooltip for interactive details
    tooltip=[
        'Ticker',
        alt.Tooltip('Portfolio Percentage Clean', title='Portfolio %', format='.1f'),
        alt.Tooltip('Ownership Performance Clean', title='Total Return %', format='.1f'),
        alt.Tooltip('Market Value Clean', title='Market Value (€)', format='$,.2f')
    ]
).properties(title='Position Size vs. Total Return by Ticker (Allocation Impact)').interactive()

# Add Ticker labels next to the scatter points for clear identification
chart_allocation_impact_final = chart_allocation_impact + chart_allocation_impact.mark_text(
    align='left', baseline='middle', dx=10, text='Ticker'
).encode(size=alt.value(0), color=alt.value('black')).interactive()

print("\n--- 4. Allocation Impact vs. Returns Scatter Plot ---")
display(chart_allocation_impact_final)

--- 1. Portfolio Allocation Pie Chart ---


alt.LayerChart(...)


--- 2. Total Returns Bar Chart ---


alt.Chart(...)


--- 3. Performance vs. Valuation Scatter Plot ---


alt.LayerChart(...)


--- 4. Allocation Impact vs. Returns Scatter Plot ---


alt.LayerChart(...)


# Portfolio Dashboard Summary

## 📊 Interactive Investment Analysis

This dashboard provides a comprehensive visual overview of your portfolio's allocation, total returns, and key performance indicators based on the provided data.

---

### 🥧 Portfolio Allocation by Ticker
The pie chart illustrates the current distribution of your investments across different tickers based on their portfolio percentage. It highlights which assets constitute the largest portions of your portfolio.

### 📈 Total Returns by Ticker
The bar chart shows the total return (ownership performance) for each ticker:
- **Green bars** indicate positive returns
- **Red bars** indicate negative returns

This visualization allows you to quickly identify the best and worst performing assets in your portfolio.

### 💹 Total Return vs. Valuation
This scatter plot visualizes the relationship between the total return and the Price-to-Earnings (P/E) ratio for each ticker. The P/E ratio is a common valuation metric that helps you understand how the performance of your assets relates to their current valuation.

### 💰 Position Size vs. Total Return
This scatter plot explores the impact of your position size (portfolio allocation) on the total return. The size of each dot represents the market value of the position, helping identify if larger positions are correlating with higher or lower returns, and the absolute value tied to those positions.

---

## ✨ Demonstrating Analytical Impact

The visualizations above are a result of my data analysis approach applied to my investment portfolio. Through careful examination of factors such as:

- Individual asset performance
- Valuation metrics (P/E ratio)
- Impact of position sizing

I have achieved a notable portfolio return. The charts visually support these outcomes, showcasing the performance of key holdings and providing insights into the strategies employed. This project demonstrates my ability to utilize data analysis techniques to understand and evaluate financial performance, a key skill for data analysis roles.